In [1]:
import os

# Set to your desired path
os.chdir('/home/pratham.doshi@simform.dom/Desktop/python training')

# Confirm current working directory
print("Current working directory:", os.getcwd())


Current working directory: /home/pratham.doshi@simform.dom/Desktop/python training


In [2]:
!pwd

/home/pratham.doshi@simform.dom/Desktop/python training


In [3]:
from conversation_bot.state_schema.graph_state import agentState
from conversation_bot.prompts.routing_prompt import router_prompt
from conversation_bot.utils_function.utils import get_llm
from conversation_bot.utils_function.logger_utility import get_logger
import traceback

logger = get_logger("RoutingNode")


def get_routing_node(llm):

    def routing_node(state: agentState) -> agentState:
        try:
            router_chain = router_prompt | llm
            result = router_chain.invoke({"query": state["query"]})
            # logger.info(f"Query for agent to be decided: {state["query"]}")
            logger.info(f"Agent selected by router: {result.content}")
            print("[QUERY]" , state["query"])

            return {"agent_used": [result.content]}

        except Exception as e:
            logger.error(f"Error in routing node: {e}")
            logger.debug(traceback.format_exc())

            # Optional: fallback to default agent
            return {"agent_used": ["llm_agent"]}

    return routing_node


In [4]:
!pwd

/home/pratham.doshi@simform.dom/Desktop/python training


In [5]:
from conversation_bot.state_schema.graph_state import agentState
from conversation_bot.prompts.input_validation_prompt import validation_prompt
from langchain_core.messages import AIMessage, ToolMessage
from conversation_bot.utils_function.logger_utility import get_logger
from langgraph.types import interrupt

logger = get_logger("ValidationNode")

def get_validation_node(llm_with_tool):

    def validation_node(state: agentState ) -> agentState:
        feedback_chain = validation_prompt | llm_with_tool


        result = feedback_chain.invoke({"query": state["query"]})

        messages = state.get("messages", [])

        # ✅ Prevent duplicate tool calls
        if isinstance(result, AIMessage):
            last_tool_call_names = [
                tc["name"]
                for m in messages
                if isinstance(m, AIMessage)
                for tc in getattr(m, "tool_calls", [])
            ]
            current_tool_calls = [tc["name"] for tc in result.tool_calls]
        
            if any(name in last_tool_call_names for name in current_tool_calls):
                print("🔁 Skipping duplicate tool call from AIMessage.")
            else:
                messages.append(result)
        else:
            messages.append(result)
        

        # ✅ If human feedback is found, use it
        for msg in messages:
            if isinstance(msg, ToolMessage) and msg.name == "human_feedback":
                print(f"✅ Received human clarification: {msg.content}")
                return {
                    "messages": messages,
                    "query": msg.content
                }

        # Default: no intervention
        print("No tool or feedback intervention. Continuing with same query.")
        print("State query to be continue" , state["query"])
        return {
            "messages": messages,
            "query": state["query"]
        }
    return validation_node


## Code for running workflow

In [6]:
import traceback
from PIL import Image
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langgraph.types import Command

from conversation_bot.utils_function.logger_utility import get_logger
from conversation_bot.state_schema.graph_state import agentState
from conversation_bot.memory.in_memory import checkpointer
from conversation_bot.edge.routing_edge import should_continue
from conversation_bot.edge.validation_edge import should_continue_human
from conversation_bot.agents.llm_search_agent import baseagent
from conversation_bot.utils_function.utils import get_llm, get_image_llm
from conversation_bot.utils_function.langgraph_utils import get_llm_with_tool
from conversation_bot.tools.human_feedback_tool import human_feedback
from conversation_bot.tools.image_gen_tool import image_tool
from conversation_bot.tools.web_search_tool import tavily_search_tool_func
from conversation_bot.prompts.system_prompt_image import image_system_prompt

logger = get_logger("Workflow")

thread_config = {"configurable": {"thread_id": "2555"}}

instruction = (
    "When generating final answer always return image path without any extras "
    "for example **generated_images/bcfc234dca554652a72a456177d97ee9.png**"
)

class workflow:
    def __init__(self):
        try:
            logger.info("Initializing workflow...")
            self.llm = get_llm()
            self.image_llm = get_image_llm()
            self.llm_With_tool = get_llm_with_tool()
            self.checkpointer = checkpointer

            self.setup_tool()
            self.setup_agents()
            self.setup_conditional_edge()
            self.setup_node()
            logger.info("Workflow initialized successfully.")
        except Exception as e:
            logger.error(f"Workflow initialization failed: {e}")
            logger.debug(traceback.format_exc())
            raise

    def setup_tool(self):
        self.image_gen_tool = image_tool(self.image_llm)
        self.human_feedback_tool = human_feedback

    def setup_agents(self):
        self.llm_agent = baseagent(self.llm, [], "llm_expert")
        self.image_agent = baseagent(self.llm, [self.image_gen_tool], "image_expert", instruction=instruction)
        self.web_agent = baseagent(self.llm, [tavily_search_tool_func], "web_expert")

    def setup_node(self):
        self.validation_node = get_validation_node(self.llm_With_tool)
        self.routing_node = get_routing_node(self.llm)

    def setup_conditional_edge(self):
        self.should_continue_routing = should_continue
        self.should_continue_validation = should_continue_human

    def create_graph(self):
        try:
            graph = StateGraph(agentState)

            graph.set_entry_point("is feedback needed")
            graph.add_node("is feedback needed", self.validation_node)
            graph.add_node("routing agent", self.routing_node)

            tool = ToolNode([self.human_feedback_tool])
            graph.add_node("human_feedback", tool)

            graph.add_conditional_edges("is feedback needed",
                                        self.should_continue_validation,
                                        {
                                            "tools": "human_feedback",
                                            "routing agent": "routing agent"
                                        })

            graph.add_edge("human_feedback", "is feedback needed")

            graph.add_conditional_edges("routing agent", self.should_continue_routing, {
                "llm_agent": "llm generation",
                "image_agent": "image generation",
                "web_agent": "web generation",
            })

            graph.add_node("llm generation", self.llm_agent)
            graph.add_node("web generation", self.web_agent)
            graph.add_node("image generation", self.image_agent)

            graph.add_edge("llm generation", END)
            graph.add_edge("web generation", END)
            graph.add_edge("image generation", END)

            logger.info("Graph created and compiled.")
            return graph.compile(checkpointer=self.checkpointer)

        except Exception as e:
            logger.error(f"Failed to create LangGraph: {e}")
            logger.debug(traceback.format_exc())
            raise


class WorkflowRunner:
    def __init__(self, workflow, thread_config):
        try:
            logger.info("Initializing WorkflowRunner...")
            self.workflow = workflow
            self.app = self.workflow.create_graph()
            self.thread_config = thread_config
            logger.info("WorkflowRunner ready.")
        except Exception as e:
            logger.error(f"WorkflowRunner initialization failed: {e}")
            logger.debug(traceback.format_exc())
            raise

    def handle_query(self, query: str):
        try:
            logger.info(f"Handling user query: {query}")
            result = self.app.invoke({"query": query}, config=self.thread_config)
            return self._handle_result(result)
        except Exception as e:
            logger.error(f"Error during query handling: {e}")
            logger.debug(traceback.format_exc())
            return {"error": str(e)}

    def resume_with_feedback(self, feedback: str):
        try:
            logger.info(f"Resuming with human feedback: {feedback}")
            result = self.app.invoke(Command(resume=feedback), config=self.thread_config)
            return self._handle_result(result)
        except Exception as e:
            logger.error(f"Error resuming with feedback: {e}")
            logger.debug(traceback.format_exc())
            return {"error": str(e)}

    def check_for_interrupt(self):
        try:
            state = self.app.get_state(config=self.thread_config)
            return state.interrupts
        except Exception as e:
            logger.error(f"Error checking for interrupts: {e}")
            logger.debug(traceback.format_exc())
            return False

    def _handle_result(self, result):
        try:
            agent_list = result.get("agent_used", [])
            last_agent = agent_list[-1] if agent_list else None
            last_message = result["messages"][-1] if "messages" in result else None

            if "__interrupt__" in result:
                logger.warning("🔁 Awaiting human feedback...")
                print(result["__interrupt__"][-1].value)

            elif last_agent == "image_agent" and "png" in last_message.content:
                image_path = last_message.content
                logger.info(f"[🖼️ IMAGE GENERATED] {image_path}")
                print("[🖼️ IMAGE GENERATED]", image_path)
                Image.open(image_path).show()

            else:
                logger.info(f"[💬 RESPONSE] {last_message.content}")
                print("[💬 Response]:", last_message)

            return result

        except Exception as e:
            logger.error(f"Error while processing result: {e}")
            logger.debug(traceback.format_exc())
            return {"error": str(e)}


# # ✅ CLI Runner
# if __name__ == "__main__":
#     try:
#         wf = workflow()
#         runner = WorkflowRunner(wf, thread_config)

#         while True:
#             query = input("Enter your query (or 'done'): ")
#             if query.lower() == "done":
#                 break

#             result = runner.handle_query(query)

#             while runner.check_for_interrupt():
#                 feedback = input("Human input required. Provide clarification: ")
#                 result = runner.resume_with_feedback(feedback)

#     except Exception as e:
#         logger.critical("Critical failure in main loop.", exc_info=True)


2025-06-12 12:30:36,158 | INFO | MongoCheckpointer | MongoDB checkpointer initialized successfully.


In [7]:

wf = workflow()
runner = WorkflowRunner(wf, thread_config)

# while True:
#     query = input("Enter your query (or 'done'): ")
#     if query.lower() == "done":
#         break

#     result = runner.handle_query(query)

#     while runner.check_for_interrupt():
#         feedback = input("Human input required. Provide clarification: ")
#         result = runner.resume_with_feedback(feedback)



2025-06-12 13:28:01,652 | INFO | Workflow | Initializing workflow...
2025-06-12 13:28:01,654 | INFO | Utils | Initializing LLM with model: gemini-2.0-flash
2025-06-12 13:28:01,677 | INFO | Utils | Initializing image LLM with model: models/gemini-2.0-flash-preview-image-generation
2025-06-12 13:28:01,680 | INFO | Utils | Initializing LLM with model: gemini-2.0-flash
2025-06-12 13:28:01,691 | INFO | LLMWithTool | LLM successfully bound with human_feedback tool.
2025-06-12 13:28:01,707 | INFO | baseagent | Agent creation successful
2025-06-12 13:28:01,711 | INFO | baseagent | Initialization done | LLM: model='models/gemini-2.0-flash' google_api_key=SecretStr('**********') client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x7f6ab315a660> default_metadata=() model_kwargs={} | Tools: [] | Name: llm_expert
2025-06-12 13:28:01,735 | INFO | baseagent | Agent creation successful
2025-06-12 13:28:01,737 | INFO | baseagent | Initializa

In [8]:
query = "What is ai "

In [9]:
# query = input("Enter your query (or 'done'): ")
result = runner.handle_query(query)
while runner.check_for_interrupt():
        feedback = input("Human input required. Provide clarification: ")
        result = runner.resume_with_feedback(feedback)

2025-06-12 13:28:37,562 | INFO | Workflow | Handling user query: What is ai 
2025-06-12 13:28:38,646 | INFO | HumanRouter | No tool_calls detected. Routing to 'routing agent'


✅ Received human clarification: Error: TypeError('exceptions must derive from BaseException')
 Please fix your mistakes.


2025-06-12 13:28:39,403 | INFO | Workflow | Agent selected by router: llm_agent
2025-06-12 13:28:39,409 | INFO | Router | Routing decision based on last agent: llm_agent
2025-06-12 13:28:39,413 | INFO | baseagent | Agent invoked for query: Error: TypeError('exceptions must derive from BaseException')
 Please fix your mistakes.


[QUERY] Error: TypeError('exceptions must derive from BaseException')
 Please fix your mistakes.


2025-06-12 13:28:42,211 | INFO | baseagent | Agent response added to state messages.
2025-06-12 13:28:42,251 | INFO | Workflow | [💬 RESPONSE] You are correct to point out the error. I apologize for the issue. As I've explained before, I cannot directly fix my code in this environment. I am a language model, and my responses are generated based on my training data and the current input. I do not have the ability to modify my own code.

I understand that you received a `TypeError('exceptions must derive from BaseException')` error. This indicates that there's a problem with how I'm handling exceptions within my code. I need to correct the way I'm trying to raise or catch errors.

To help me analyze how I triggered this error, could you please provide the history of our conversation leading up to the error, starting with the query that initiated the conversation? This will help me understand the flow of events and identify any specific actions that might have caused the problem.

With thi

[💬 Response]: content="You are correct to point out the error. I apologize for the issue. As I've explained before, I cannot directly fix my code in this environment. I am a language model, and my responses are generated based on my training data and the current input. I do not have the ability to modify my own code.\n\nI understand that you received a `TypeError('exceptions must derive from BaseException')` error. This indicates that there's a problem with how I'm handling exceptions within my code. I need to correct the way I'm trying to raise or catch errors.\n\nTo help me analyze how I triggered this error, could you please provide the history of our conversation leading up to the error, starting with the query that initiated the conversation? This will help me understand the flow of events and identify any specific actions that might have caused the problem.\n\nWith this information, I can analyze my response and provide more specific feedback to the development team to help them 